# NCAA March Madness Prediction — Kaggle Silver Medal (Top 5%)

**Competition:** [March Machine Learning Mania](https://www.kaggle.com/competitions/march-machine-learning-mania-2025)  
**Result:** 🥈 Silver Medal — Top 5% of all submissions  
**Task:** Predict win probabilities for every possible NCAA tournament matchup (men's + women's)  
**Metric:** Brier Score (lower = better-calibrated probabilities)

---

## Approach Overview

| Stage | Detail |
|---|---|
| **Feature engineering** | 22 per-team features across 7 signal groups |
| **Model** | Soft-voting ensemble: Logistic Regression + XGBoost |
| **Calibration** | Platt scaling via `CalibratedClassifierCV` |
| **Evaluation** | 5-fold cross-validated Brier score + 2025 holdout backtest |

### Feature Groups
1. **Elo ratings** — multi-season with mean reversion + home-court adjustment  
2. **SOS-adjusted margins** — full season & last-10-game momentum  
3. **Win rates** — overall, home/away/neutral splits, strength of schedule  
4. **Efficiency** — offensive/defensive ratings, net rating, margin volatility, clutch rate  
5. **Late-season form** — trajectory across season thirds  
6. **Detailed box-score stats** — rebound rates, free-throw rate & accuracy  
7. **Conference strength** — exponentially-decayed historical tournament seed quality  
8. **Massey ordinal rankings** — consensus of all ranking systems on final pre-tourney day


## 1. Imports & Constants

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline

# ── Elo hyperparameters ──────────────────────────────────────────────────────
ELO_K      = 32    # K-factor: how much each game shifts ratings
ELO_INIT   = 1500  # Starting Elo for all teams
ELO_HCA    = 65    # Home-court advantage in Elo points
REVERSION  = 0.75  # Carry over 75% of prior-season rating; 25% reverts to 1500


## 2. Data Loading

Reads all Kaggle competition CSVs from a local directory.  
Expected files: `MTeams`, `WTeams`, `MRegularSeasonCompactResults`,
`WRegularSeasonCompactResults`, `MNCAATourneyCompactResults`,
`WNCAATourneyCompactResults`, `MNCAATourneySeeds`, `WNCAATourneySeeds`,
`SampleSubmissionStage1`, `MMasseyOrdinals`, `MTeamConferences`,
`WTeamConferences`, `MRegularSeasonDetailedResults`,
`WRegularSeasonDetailedResults`.


In [ ]:
def load_competition_data(data_dir: str) -> dict:
    """Load all March Madness competition CSV files.

    Parameters
    ----------
    data_dir : str
        Path to the directory containing the Kaggle CSV files.

    Returns
    -------
    dict
        Dictionary of DataFrames keyed by short names, plus a summary dict
        printed to stdout.
    """
    data = {}
    data['m_teams']              = pd.read_csv(f'{data_dir}/MTeams.csv')
    data['w_teams']              = pd.read_csv(f'{data_dir}/WTeams.csv')
    data['m_regular']            = pd.read_csv(f'{data_dir}/MRegularSeasonCompactResults.csv')
    data['w_regular']            = pd.read_csv(f'{data_dir}/WRegularSeasonCompactResults.csv')
    data['m_tourney']            = pd.read_csv(f'{data_dir}/MNCAATourneyCompactResults.csv')
    data['w_tourney']            = pd.read_csv(f'{data_dir}/WNCAATourneyCompactResults.csv')
    data['m_seeds']              = pd.read_csv(f'{data_dir}/MNCAATourneySeeds.csv')
    data['w_seeds']              = pd.read_csv(f'{data_dir}/WNCAATourneySeeds.csv')
    data['sample_sub']           = pd.read_csv(f'{data_dir}/SampleSubmissionStage1.csv')
    data['rankings']             = pd.read_csv(f'{data_dir}/MMasseyOrdinals.csv')
    data['m_conferences']        = pd.read_csv(f'{data_dir}/MTeamConferences.csv')
    data['w_conferences']        = pd.read_csv(f'{data_dir}/WTeamConferences.csv')
    data['m_regular_detailed']   = pd.read_csv(f'{data_dir}/MRegularSeasonDetailedResults.csv')
    data['w_regular_detailed']   = pd.read_csv(f'{data_dir}/WRegularSeasonDetailedResults.csv')

    summary = {
        'status':               'success',
        'seasons':              f"{data['m_regular']['Season'].min()}–{data['m_regular']['Season'].max()}",
        'mens_teams':           len(data['m_teams']),
        'womens_teams':         len(data['w_teams']),
        'regular_season_games': len(data['m_regular']) + len(data['w_regular']),
        'tourney_games':        len(data['m_tourney']) + len(data['w_tourney']),
        'submission_rows':      len(data['sample_sub']),
        'message':              'All data loaded. Ready for feature engineering.',
    }
    print(summary)
    return data


## 3. Feature Engineering

### 3.1 Elo Ratings with Mean Reversion

Elo is computed game-by-game across every regular season for both men's and
women's teams.  Between seasons, each team's rating reverts 25% toward 1500
so that dominant programs don't accumulate infinite rating.  Home-court
advantage is worth +65 Elo points when computing expected win probability.


In [ ]:
def compute_elo_ratings(data: dict) -> dict:
    """Compute Elo ratings with inter-season mean reversion.

    Returns
    -------
    dict
        Keys are (season, team_id) tuples; values are end-of-season Elo floats.
    """
    elo: dict = {}
    last_season_ending_elo: dict = {}

    print("Computing Elo ratings with mean reversion...")
    for gender in ['m', 'w']:
        df = data[f'{gender}_regular'].copy()
        seasons = sorted(df['Season'].unique())

        for season in seasons:
            season_df = df[df['Season'] == season].sort_values('DayNum')

            # Inter-season reversion: pull each team 25% back toward 1500
            current_season_elos = {
                team_id: (old_elo * REVERSION) + (ELO_INIT * (1 - REVERSION))
                for team_id, old_elo in last_season_ending_elo.items()
            }

            for _, row in season_df.iterrows():
                w_id, l_id = row['WTeamID'], row['LTeamID']
                r_w = current_season_elos.get(w_id, ELO_INIT)
                r_l = current_season_elos.get(l_id, ELO_INIT)

                # Home-court adjustment: +HCA for home winner, -HCA for away winner
                hca = ELO_HCA if row['WLoc'] == 'H' else (-ELO_HCA if row['WLoc'] == 'A' else 0)
                exp_w = 1 / (1 + 10 ** ((r_l - (r_w + hca)) / 400))

                shift = ELO_K * (1 - exp_w)
                current_season_elos[w_id] = r_w + shift
                current_season_elos[l_id] = r_l - shift

            for team_id, final_elo in current_season_elos.items():
                elo[(season, team_id)] = final_elo
                last_season_ending_elo[team_id] = final_elo

    print(f"Elo computed for {len(elo):,} team-season pairs.")
    return elo


### 3.2 SOS-Adjusted Point Differentials & Momentum

Raw margin is adjusted by opponent strength:
`adj_margin = avg_margin + (avg_opp_elo − 1500) / 10`

`momentum` applies the same formula to only the final 10 regular-season games,
capturing late-season form.


In [ ]:
def calculate_point_differential(data: dict, elo: dict) -> dict:
    """Season-long and last-10-game SOS-adjusted scoring margins.

    Returns
    -------
    dict
        Keys are (season, team_id); values contain 'adj_margin' and 'momentum'.
    """
    print("Calculating SOS-adjusted point differentials...")
    stats_map: dict = {}

    for gender in ['m', 'w']:
        df = data[f'{gender}_regular'].copy().sort_values(['Season', 'DayNum'])
        all_teams = df[['WTeamID', 'LTeamID']].stack().unique()

        for season in df['Season'].unique():
            s_df = df[df['Season'] == season]

            for team in all_teams:
                t_games = s_df[(s_df['WTeamID'] == team) | (s_df['LTeamID'] == team)].copy()
                if t_games.empty:
                    continue

                t_games['MyMargin'] = t_games.apply(
                    lambda r: (r['WScore'] - r['LScore']) if r['WTeamID'] == team
                              else (r['LScore'] - r['WScore']), axis=1)
                t_games['OppID'] = t_games.apply(
                    lambda r: r['LTeamID'] if r['WTeamID'] == team else r['WTeamID'], axis=1)
                t_games['OppElo'] = t_games['OppID'].apply(
                    lambda opp: elo.get((season, opp), ELO_INIT))

                full_margin  = t_games['MyMargin'].mean()
                full_opp_elo = t_games['OppElo'].mean()
                adj_full     = full_margin + (full_opp_elo - ELO_INIT) / 10.0

                last_10      = t_games.tail(10)
                l10_margin   = last_10['MyMargin'].mean()
                l10_opp_elo  = last_10['OppElo'].mean()
                adj_l10      = l10_margin + (l10_opp_elo - ELO_INIT) / 10.0

                stats_map[(season, team)] = {'adj_margin': adj_full, 'momentum': adj_l10}

    return stats_map


### 3.3 Win Rate, Location Splits & Strength of Schedule

Captures overall win%, home/away/neutral splits, and a KenPom-style SOS
metric (average opponent win%).


In [ ]:
def calculate_win_rate_and_sos(data: dict) -> dict:
    """Win%, home/away/neutral splits, and SOS (opponent win%).

    Returns
    -------
    dict
        Keys are (season, team_id); values contain win_pct, home_win_pct,
        away_win_pct, neut_win_pct, sos, n_games.
    """
    stats_map: dict = {}

    for gender in ['m', 'w']:
        df = data[f'{gender}_regular'].copy()

        for season in df['Season'].unique():
            s_df      = df[df['Season'] == season]
            all_teams = pd.concat([s_df['WTeamID'], s_df['LTeamID']]).unique()

            for team in all_teams:
                wins   = s_df[s_df['WTeamID'] == team]
                losses = s_df[s_df['LTeamID'] == team]
                n_wins, n_losses = len(wins), len(losses)
                n_games = n_wins + n_losses
                if n_games == 0:
                    continue

                win_pct = n_wins / n_games

                # Location splits (WLoc is from the winner's perspective)
                home_wins   = len(wins[wins['WLoc'] == 'H'])
                away_wins   = len(wins[wins['WLoc'] == 'A'])
                home_losses = len(losses[losses['WLoc'] == 'A'])  # winner was away → team was home
                away_losses = len(losses[losses['WLoc'] == 'H'])  # winner was home → team was away
                home_games  = home_wins + home_losses
                away_games  = away_wins + away_losses
                neut_games  = n_games - home_games - away_games

                home_win_pct = home_wins / home_games if home_games > 0 else win_pct
                away_win_pct = away_wins / away_games if away_games > 0 else win_pct
                neut_win_pct = (n_wins - home_wins - away_wins) / neut_games if neut_games > 0 else win_pct

                # SOS: average win% of all opponents
                opp_ids = pd.concat([wins['LTeamID'], losses['WTeamID']]).values
                opp_win_pcts = []
                for opp in opp_ids:
                    opp_w = len(s_df[s_df['WTeamID'] == opp])
                    opp_l = len(s_df[s_df['LTeamID'] == opp])
                    opp_g = opp_w + opp_l
                    if opp_g > 0:
                        opp_win_pcts.append(opp_w / opp_g)
                sos = np.mean(opp_win_pcts) if opp_win_pcts else 0.5

                stats_map[(season, team)] = {
                    'win_pct':      win_pct,
                    'home_win_pct': home_win_pct,
                    'away_win_pct': away_win_pct,
                    'neut_win_pct': neut_win_pct,
                    'sos':          sos,
                    'n_games':      n_games,
                }

    return stats_map


### 3.4 Offensive / Defensive Efficiency

Points scored and allowed per game, net rating, margin volatility (std dev),
and clutch win rate (games decided by ≤5 points).


In [ ]:
def calculate_efficiency(data: dict) -> dict:
    """Offensive/defensive ratings, net rating, volatility, and clutch rate.

    Returns
    -------
    dict
        Keys are (season, team_id); values contain off_rating, def_rating,
        net_rating, margin_std, clutch_rate.
    """
    stats_map: dict = {}

    for gender in ['m', 'w']:
        df = data[f'{gender}_regular'].copy()

        for season in df['Season'].unique():
            s_df      = df[df['Season'] == season]
            all_teams = pd.concat([s_df['WTeamID'], s_df['LTeamID']]).unique()

            for team in all_teams:
                wins    = s_df[s_df['WTeamID'] == team]
                losses  = s_df[s_df['LTeamID'] == team]
                n_games = len(wins) + len(losses)
                if n_games == 0:
                    continue

                pts_scored  = wins['WScore'].sum()  + losses['LScore'].sum()
                pts_allowed = wins['LScore'].sum()  + losses['WScore'].sum()
                off_rating  = pts_scored  / n_games
                def_rating  = pts_allowed / n_games
                net_rating  = off_rating  - def_rating

                margins = pd.concat([
                    wins['WScore']   - wins['LScore'],
                    losses['LScore'] - losses['WScore'],
                ])
                margin_std  = margins.std()

                close_wins   = len(wins[wins['WScore']   - wins['LScore']   <= 5])
                close_losses = len(losses[losses['WScore'] - losses['LScore'] <= 5])
                close_games  = close_wins + close_losses
                clutch_rate  = close_wins / close_games if close_games > 0 else 0.5

                stats_map[(season, team)] = {
                    'off_rating':  off_rating,
                    'def_rating':  def_rating,
                    'net_rating':  net_rating,
                    'margin_std':  margin_std,
                    'clutch_rate': clutch_rate,
                }

    return stats_map


### 3.5 Late-Season Form & Trajectory

Splits each season into thirds (early / mid / late) and computes win% per
period.  `trajectory = late_win_pct − early_win_pct` — positive means the
team is peaking into the tournament.


In [ ]:
def calculate_late_season_form(data: dict) -> dict:
    """Win% by season third and improvement trajectory.

    Returns
    -------
    dict
        Keys are (season, team_id); values contain early_win_pct, mid_win_pct,
        late_win_pct, trajectory.
    """
    stats_map: dict = {}

    for gender in ['m', 'w']:
        df = data[f'{gender}_regular'].copy()

        for season in df['Season'].unique():
            s_df     = df[df['Season'] == season]
            day_min, day_max = s_df['DayNum'].min(), s_df['DayNum'].max()
            third    = (day_max - day_min) / 3
            early_cutoff = day_min + third
            late_cutoff  = day_max - third
            all_teams    = pd.concat([s_df['WTeamID'], s_df['LTeamID']]).unique()

            for team in all_teams:
                t_df = s_df[(s_df['WTeamID'] == team) | (s_df['LTeamID'] == team)]
                if t_df.empty:
                    continue

                def win_pct_in(subset):
                    w = len(subset[subset['WTeamID'] == team])
                    return w / len(subset) if len(subset) > 0 else None

                early    = t_df[t_df['DayNum'] <= early_cutoff]
                mid      = t_df[(t_df['DayNum'] > early_cutoff) & (t_df['DayNum'] <= late_cutoff)]
                late     = t_df[t_df['DayNum'] > late_cutoff]
                early_wp = win_pct_in(early)
                late_wp  = win_pct_in(late)
                trajectory = (late_wp - early_wp) if (late_wp is not None and early_wp is not None) else 0.0

                stats_map[(season, team)] = {
                    'early_win_pct': early_wp or 0.5,
                    'mid_win_pct':   win_pct_in(mid) or 0.5,
                    'late_win_pct':  late_wp or 0.5,
                    'trajectory':    trajectory,
                }

    return stats_map


### 3.6 Box-Score Stats: Rebound Rates & Free Throws

- **OR rate** = team offensive rebounds / (team OR + opponent DR)  
- **DR rate** = team defensive rebounds / (team DR + opponent OR)  
- **FT rate** = FTA / FGA (how often a team gets to the line)  
- **FT%** = FTM / FTA


In [ ]:
def calculate_detailed_stats(data: dict) -> dict:
    """Rebound rates and free-throw stats from detailed game logs.

    Returns
    -------
    dict
        Keys are (season, team_id); values contain or_rate, dr_rate,
        ft_rate, ft_pct.
    """
    print("Computing rebound rates and free-throw stats...")
    stats_map: dict = {}

    for gender in ['m', 'w']:
        df = data[f'{gender}_regular_detailed'].copy()

        for season in df['Season'].unique():
            s_df      = df[df['Season'] == season]
            all_teams = pd.concat([s_df['WTeamID'], s_df['LTeamID']]).unique()

            for team in all_teams:
                wins   = s_df[s_df['WTeamID'] == team]
                losses = s_df[s_df['LTeamID'] == team]

                team_or = wins['WOR'].sum()  + losses['LOR'].sum()
                opp_dr  = wins['LDR'].sum()  + losses['WDR'].sum()
                team_dr = wins['WDR'].sum()  + losses['LDR'].sum()
                opp_or  = wins['LOR'].sum()  + losses['WOR'].sum()
                or_rate = team_or / (team_or + opp_dr) if (team_or + opp_dr) > 0 else 0.30
                dr_rate = team_dr / (team_dr + opp_or) if (team_dr + opp_or) > 0 else 0.70

                team_fta = wins['WFTA'].sum() + losses['LFTA'].sum()
                team_ftm = wins['WFTM'].sum() + losses['LFTM'].sum()
                team_fga = wins['WFGA'].sum() + losses['LFGA'].sum()
                ft_rate  = team_fta / team_fga if team_fga > 0 else 0.33
                ft_pct   = team_ftm / team_fta if team_fta > 0 else 0.70

                stats_map[(season, team)] = {
                    'or_rate': or_rate,
                    'dr_rate': dr_rate,
                    'ft_rate': ft_rate,
                    'ft_pct':  ft_pct,
                }

    print(f"Detailed stats computed for {len(stats_map):,} team-season pairs.")
    return stats_map


### 3.7 Conference Strength (Exponentially Decayed)

For each team's conference, computes a weighted average of historical
tournament seed quality and team depth using exponential decay
(`weight = 0.7 ^ years_ago`) so that recent seasons count more.


In [ ]:
def compute_conference_strength(data: dict) -> dict:
    """Weighted historical tournament seed quality per conference.

    Uses exponential decay (DECAY=0.7 per year) so recent seasons
    are weighted more heavily than distant ones.

    Returns
    -------
    dict
        Keys are (season, team_id); values contain conf_seed_str
        (lower = stronger conference) and conf_depth (avg tourney teams).
    """
    print("Computing conference strength...")
    DECAY = 0.7

    # (season, team_id) → seed number
    seed_map: dict = {}
    for df_key in ['m_seeds', 'w_seeds']:
        for _, row in data[df_key].iterrows():
            seed_map[(int(row['Season']), int(row['TeamID']))] = int(row['Seed'][1:3])

    # (season, team_id) → conference abbreviation
    conf_map: dict = {}
    for df_key in ['m_conferences', 'w_conferences']:
        for _, row in data[df_key].iterrows():
            conf_map[(int(row['Season']), int(row['TeamID']))] = row['ConfAbbrev']

    # conf_history[(season, conf)] → {'avg_seed': float, 'count': int}
    conf_history: dict = {}
    all_seasons = sorted({s for s, _ in conf_map})

    for season in all_seasons:
        conf_teams: dict = {}
        for (s, tid), conf in conf_map.items():
            if s == season:
                conf_teams.setdefault(conf, []).append(tid)

        for conf, teams in conf_teams.items():
            seeds = [seed_map[(season, t)] for t in teams if (season, t) in seed_map]
            if seeds:
                conf_history[(season, conf)] = {'avg_seed': np.mean(seeds), 'count': len(seeds)}

    stats_map: dict = {}
    for season in all_seasons:
        for (s, tid), conf in conf_map.items():
            if s != season:
                continue
            w_seed_sum = w_count_sum = total_w = 0.0
            for past in all_seasons:
                if past >= season:
                    break
                w      = DECAY ** (season - past)
                hist   = conf_history.get((past, conf))
                if hist:
                    w_seed_sum  += w * hist['avg_seed']
                    w_count_sum += w * hist['count']
                    total_w     += w
            if total_w > 0:
                conf_avg_seed  = w_seed_sum  / total_w
                conf_avg_depth = w_count_sum / total_w
            else:
                conf_avg_seed, conf_avg_depth = 8.5, 2.0

            stats_map[(season, tid)] = {
                'conf_seed_str': conf_avg_seed,
                'conf_depth':    conf_avg_depth,
            }

    print(f"Conference strength computed for {len(stats_map):,} team-season pairs.")
    return stats_map


### 3.8 Massey Consensus Rankings

Averages ordinal rank across all external ranking systems on the final
pre-tournament day.  Men's only (falls back to seed-derived rank for women).


In [ ]:
def compute_massey_rankings(data: dict) -> dict:
    """Average Massey ordinal rank across all systems on the last ranking day.

    Returns
    -------
    dict
        Keys are (season, team_id); values are average ordinal rank (float).
        Lower rank = better team.
    """
    print("Computing Massey consensus rankings...")
    rankings = data['rankings']
    rank_map: dict = {}

    for season, sdf in rankings.groupby('Season'):
        max_day  = sdf['RankingDayNum'].max()
        final    = sdf[sdf['RankingDayNum'] == max_day]
        avg_ranks = final.groupby('TeamID')['OrdinalRank'].mean()
        for team_id, avg_rank in avg_ranks.items():
            rank_map[(season, team_id)] = avg_rank

    print(f"Massey rankings computed for {len(rank_map):,} team-season pairs.")
    return rank_map


## 4. Matchup Feature Vector

All 22 features are expressed as **Team A minus Team B** differences so the
model sees a symmetric, sign-consistent representation regardless of which
team is listed first.  The convention is always lower TeamID = Team 1.


In [ ]:
FEATURE_NAMES = [
    'elo_diff',          'elo_curr_diff',    'seed_diff',
    'margin_diff',       'momentum_diff',
    'win_pct_diff',      'neut_win_pct_diff','sos_diff',
    'off_rating_diff',   'def_rating_diff',  'net_rating_diff',
    'margin_std_diff',   'clutch_rate_diff',
    'late_win_pct_diff', 'trajectory_diff',  'massey_rank_diff',
    'conf_seed_str_diff','conf_depth_diff',
    'or_rate_diff',      'dr_rate_diff',
    'ft_rate_diff',      'ft_pct_diff',
]

def build_matchup_features(
    season, t1_id, t2_id,
    seed_map, elo, point_diff,
    win_rate_and_sos, efficiency, late_season_form,
    massey_rankings, conf_strength, detailed_stats
) -> list:
    """Build the 22-element feature vector for a single matchup.

    Parameters
    ----------
    season : int
    t1_id : int   Lower TeamID (convention)
    t2_id : int   Higher TeamID (convention)
    ... feature dicts produced by the functions above

    Returns
    -------
    list of float (length 22)
    """
    pd_default   = {'adj_margin': 0,    'momentum': 0}
    wr_default   = {'win_pct': 0.5,     'neut_win_pct': 0.5, 'sos': 0.5}
    eff_default  = {'off_rating': 70,   'def_rating': 70, 'net_rating': 0,
                    'margin_std': 10,   'clutch_rate': 0.5}
    lsf_default  = {'late_win_pct': 0.5,'trajectory': 0}
    conf_default = {'conf_seed_str': 8.5,'conf_depth': 2.0}
    ds_default   = {'or_rate': 0.3,     'dr_rate': 0.7, 'ft_rate': 0.33, 'ft_pct': 0.70}
    median_rank  = 176.0

    def _build(tid):
        seed = seed_map.get((season, tid), 8)
        return {
            **point_diff.get((season, tid), pd_default),
            **win_rate_and_sos.get((season, tid), wr_default),
            **efficiency.get((season, tid), eff_default),
            **late_season_form.get((season, tid), lsf_default),
            **conf_strength.get((season, tid), conf_default),
            **detailed_stats.get((season, tid), ds_default),
            'elo_prev':    elo.get((season - 1, tid), ELO_INIT),
            'elo_curr':    elo.get((season,     tid), ELO_INIT),
            'seed':        seed,
            'massey_rank': massey_rankings.get((season, tid), seed * (median_rank / 8)),
        }

    t1, t2 = _build(t1_id), _build(t2_id)

    return [
        t1['elo_prev']    - t2['elo_prev'],
        t1['elo_curr']    - t2['elo_curr'],
        t2['seed']        - t1['seed'],          # lower seed number = better team
        t1['adj_margin']  - t2['adj_margin'],
        t1['momentum']    - t2['momentum'],
        t1['win_pct']     - t2['win_pct'],
        t1['neut_win_pct']- t2['neut_win_pct'],
        t1['sos']         - t2['sos'],
        t1['off_rating']  - t2['off_rating'],
        t1['def_rating']  - t2['def_rating'],
        t1['net_rating']  - t2['net_rating'],
        t1['margin_std']  - t2['margin_std'],
        t1['clutch_rate'] - t2['clutch_rate'],
        t1['late_win_pct']- t2['late_win_pct'],
        t1['trajectory']  - t2['trajectory'],
        t2['massey_rank'] - t1['massey_rank'],   # higher rank number = worse team
        t2['conf_seed_str']- t1['conf_seed_str'],
        t1['conf_depth']  - t2['conf_depth'],
        t1['or_rate']     - t2['or_rate'],
        t1['dr_rate']     - t2['dr_rate'],
        t1['ft_rate']     - t2['ft_rate'],
        t1['ft_pct']      - t2['ft_pct'],
    ]


## 5. Model Training

### Architecture

```
Raw features (22)
       │
       ├─── StandardScaler → LogisticRegression (C=1.0)
       │
       └─── XGBClassifier (depth=2, lr=0.02, n=10)
                │
           Soft-voting ensemble (equal weights)
                │
           CalibratedClassifierCV (Platt sigmoid, 5-fold CV)
```

**Why this ensemble?**
- Logistic regression captures the strong linear signals (Elo, seed difference).
- Shallow XGBoost captures nonlinear interactions (e.g. good offense × weak opponent defense).
- Platt scaling calibrates the raw probabilities so the Brier score is optimised
  even when the ensemble is overconfident.


In [ ]:
def train_prediction_model(
    data, elo, point_diff, win_rate_and_sos,
    efficiency, late_season_form, massey_rankings,
    conf_strength, detailed_stats
) -> dict:
    """Train a calibrated LR + XGBoost ensemble on historical tournament games.

    Training data: all tournament games from 2003 onward (men's + women's).
    Label convention: 1 if lower-ID team won, 0 otherwise.

    Returns
    -------
    dict with keys: model, seed_map, training_games, cv_brier_score,
                    xgb_feature_importance, lr_coefficients, message.
    """
    # Build seed lookup covering both genders
    seed_map: dict = {}
    for df_key in ['m_seeds', 'w_seeds']:
        for _, row in data[df_key].iterrows():
            seed_map[(int(row['Season']), int(row['TeamID']))] = int(row['Seed'][1:3])

    X, y = [], []
    for t_key in ['m_tourney', 'w_tourney']:
        for _, row in data[t_key].iterrows():
            season      = int(row['Season'])
            if season < 2003:
                continue
            w_id, l_id = int(row['WTeamID']), int(row['LTeamID'])
            t1, t2      = (w_id, l_id) if w_id < l_id else (l_id, w_id)
            label       = 1 if w_id < l_id else 0

            feat = build_matchup_features(
                season, t1, t2, seed_map, elo, point_diff,
                win_rate_and_sos, efficiency, late_season_form,
                massey_rankings, conf_strength, detailed_stats
            )
            X.append(feat)
            y.append(label)

    X, y = np.array(X), np.array(y)

    # ── Base models ──────────────────────────────────────────────────────────
    lr_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('lr', LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', random_state=42)),
    ])
    xgb = XGBClassifier(
        n_estimators=10, max_depth=2, learning_rate=0.02,
        subsample=0.8, colsample_bytree=0.8,
        objective='binary:logistic', random_state=42, eval_metric='logloss',
    )

    # ── Soft-voting ensemble ─────────────────────────────────────────────────
    ensemble = VotingClassifier(
        estimators=[('lr', lr_pipeline), ('xgb', xgb)],
        voting='soft', weights=[1, 1],
    )
    ensemble.fit(X, y)

    # ── Platt calibration ────────────────────────────────────────────────────
    calibrated = CalibratedClassifierCV(ensemble, method='sigmoid', cv=5)
    calibrated.fit(X, y)

    # ── Evaluate ─────────────────────────────────────────────────────────────
    cv_scores = cross_val_score(calibrated, X, y, scoring='neg_brier_score', cv=5)
    brier     = -cv_scores.mean()

    xgb_importances = ensemble.named_estimators_['xgb'].feature_importances_
    lr_coefficients = ensemble.named_estimators_['lr'].named_steps['lr'].coef_[0]

    return {
        'status':                 'success',
        'model':                  calibrated,
        'seed_map':               seed_map,
        'training_games':         len(y),
        'cv_brier_score':         f"{brier:.4f}",
        'xgb_feature_importance': dict(zip(FEATURE_NAMES, xgb_importances.round(4))),
        'lr_coefficients':        dict(zip(FEATURE_NAMES, lr_coefficients.round(4))),
        'message':                f'Ensemble (LR + XGBoost) with Platt scaling. CV Brier: {brier:.4f}',
    }


## 6. Run Full Pipeline

In [ ]:
DATA_DIR = '.'   # ← update to your local data path

data        = load_competition_data(DATA_DIR)
elo         = compute_elo_ratings(data)
point_diff  = calculate_point_differential(data, elo)
win_rate_sos= calculate_win_rate_and_sos(data)
eff         = calculate_efficiency(data)
late_form   = calculate_late_season_form(data)
massey      = compute_massey_rankings(data)
conf_str    = compute_conference_strength(data)
det_stats   = calculate_detailed_stats(data)

result = train_prediction_model(
    data, elo, point_diff, win_rate_sos,
    eff, late_form, massey, conf_str, det_stats
)

print(f"\nTraining games : {result['training_games']:,}")
print(f"CV Brier Score  : {result['cv_brier_score']}")
print(f"\n{result['message']}")


## 7. Feature Importance & Coefficients

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# XGBoost importances
xgb_items = sorted(result['xgb_feature_importance'].items(), key=lambda x: x[1], reverse=True)
axes[0].barh([x[0] for x in xgb_items], [x[1] for x in xgb_items], color='steelblue')
axes[0].invert_yaxis()
axes[0].set_title('XGBoost Feature Importance', fontsize=13)
axes[0].set_xlabel('Importance')

# LR coefficients (absolute value)
lr_items = sorted(result['lr_coefficients'].items(), key=lambda x: abs(x[1]), reverse=True)
colors   = ['steelblue' if v > 0 else 'tomato' for _, v in lr_items]
axes[1].barh([x[0] for x in lr_items], [x[1] for x in lr_items], color=colors)
axes[1].invert_yaxis()
axes[1].set_title('Logistic Regression Coefficients\n(blue = favours Team 1)', fontsize=13)
axes[1].set_xlabel('Coefficient')
axes[1].axvline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()


## 8. 2025 Holdout Backtest

Train strictly on seasons < 2025, then evaluate on actual 2025 tournament
outcomes.  This mirrors real competition conditions and provides an
unbiased Brier score estimate.


In [ ]:
BACKTEST_SEASON = 2025
bt_seed_map     = result['seed_map']

bt_X_train, bt_y_train = [], []
bt_X_test,  bt_y_test  = [], []
bt_test_games           = []

for t_key in ['m_tourney', 'w_tourney']:
    for _, row in data[t_key].iterrows():
        season     = int(row['Season'])
        if season < 2003:
            continue
        w_id, l_id = int(row['WTeamID']), int(row['LTeamID'])
        t1, t2     = min(w_id, l_id), max(w_id, l_id)
        label      = 1 if w_id < l_id else 0

        feat = build_matchup_features(
            season, t1, t2, bt_seed_map, elo, point_diff,
            win_rate_sos, eff, late_form, massey, conf_str, det_stats
        )
        if season < BACKTEST_SEASON:
            bt_X_train.append(feat); bt_y_train.append(label)
        elif season == BACKTEST_SEASON:
            bt_X_test.append(feat);  bt_y_test.append(label)
            bt_test_games.append((w_id, l_id))

bt_X_train = np.array(bt_X_train); bt_y_train = np.array(bt_y_train)
bt_X_test  = np.array(bt_X_test);  bt_y_test  = np.array(bt_y_test)

# Re-train backtest model on <2025 data only
bt_lr  = Pipeline([('scaler', StandardScaler()),
                   ('lr', LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', random_state=42))])
bt_xgb = XGBClassifier(n_estimators=10, max_depth=2, learning_rate=0.02, subsample=0.8,
                        colsample_bytree=0.8, objective='binary:logistic',
                        random_state=42, eval_metric='logloss')
bt_ensemble  = VotingClassifier(estimators=[('lr', bt_lr), ('xgb', bt_xgb)],
                                voting='soft', weights=[1, 1])
bt_ensemble.fit(bt_X_train, bt_y_train)
bt_calibrated = CalibratedClassifierCV(bt_ensemble, method='sigmoid', cv=5)
bt_calibrated.fit(bt_X_train, bt_y_train)

bt_probs = bt_calibrated.predict_proba(bt_X_test)[:, 1]
bt_brier = np.mean((bt_probs - bt_y_test) ** 2)

print(f"2025 Holdout Brier Score : {bt_brier:.4f}")
print(f"Games predicted          : {len(bt_y_test)}")
print(f"Prediction range         : [{bt_probs.min():.4f}, {bt_probs.max():.4f}]")


In [ ]:
# Game-by-game results
team_names = {**{int(r['TeamID']): r['TeamName'] for _, r in data['m_teams'].iterrows()},
              **{int(r['TeamID']): r['TeamName'] for _, r in data['w_teams'].iterrows()}}

print("\n2025 Tournament — game-by-game predictions:")
for i, (w_id, l_id) in enumerate(bt_test_games):
    t1, t2   = min(w_id, l_id), max(w_id, l_id)
    prob     = bt_probs[i]
    actual   = bt_y_test[i]
    correct  = "✅" if (prob > 0.5) == (actual == 1) else "❌"
    t1_name  = team_names.get(t1, str(t1))
    t2_name  = team_names.get(t2, str(t2))
    winner   = team_names.get(w_id, str(w_id))
    print(f"  {correct}  {t1_name:20s} vs {t2_name:20s}  |  "
          f"Pred: {t1_name} {prob:.1%}  |  Actual winner: {winner}")


## 9. Generate Stage 2 Submission

In [ ]:
model    = result['model']
seed_map = result['seed_map']

sub   = pd.read_csv(f'{DATA_DIR}/SampleSubmissionStage2.csv')
parts = sub['ID'].str.split('_', expand=True).astype(int)

X_sub = np.array([
    build_matchup_features(
        parts[0][i], parts[1][i], parts[2][i],
        seed_map, elo, point_diff,
        win_rate_sos, eff, late_form, massey, conf_str, det_stats
    )
    for i in range(len(sub))
])

sub['Pred'] = model.predict_proba(X_sub)[:, 1]
sub.to_csv(f'{DATA_DIR}/submission_stage2.csv', index=False)

print(f"Submission saved: {len(sub):,} matchups")
print(f"Pred range  : [{sub['Pred'].min():.4f}, {sub['Pred'].max():.4f}]")
print(f"Pred mean   : {sub['Pred'].mean():.4f}")


## 10. Prediction Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(sub['Pred'], bins=50, edgecolor='white', color='steelblue', alpha=0.85)
ax.axvline(0.5, color='tomato', linestyle='--', linewidth=1.5, label='50% line')
ax.set_xlabel('Predicted win probability (Team 1)')
ax.set_ylabel('Matchup count')
ax.set_title('Distribution of predicted win probabilities — Stage 2 submission')
ax.legend()
plt.tight_layout()
plt.savefig('prediction_distribution.png', dpi=120, bbox_inches='tight')
plt.show()


## 11. Tournament-Only Matchup Analysis

In [ ]:
team_names = {**{int(r['TeamID']): r['TeamName'] for _, r in data['m_teams'].iterrows()},
              **{int(r['TeamID']): r['TeamName'] for _, r in data['w_teams'].iterrows()}}

tourney_seeds_2026 = {
    int(row['TeamID']): row['Seed']
    for _, row in pd.concat([data['m_seeds'], data['w_seeds']]).iterrows()
    if int(row['Season']) == 2026
}
tourney_ids = set(tourney_seeds_2026)

readable = sub.copy()
readable[['Season', 'Team1ID', 'Team2ID']] = (
    readable['ID'].str.split('_', expand=True).astype(int).rename(columns={0:'Season',1:'Team1ID',2:'Team2ID'})
)
readable['Team1']  = readable['Team1ID'].map(team_names)
readable['Team2']  = readable['Team2ID'].map(team_names)

tourney = readable[
    readable['Team1ID'].isin(tourney_ids) & readable['Team2ID'].isin(tourney_ids)
].copy()
tourney['Seed1']      = tourney['Team1ID'].map(tourney_seeds_2026)
tourney['Seed2']      = tourney['Team2ID'].map(tourney_seeds_2026)
tourney['Favored']    = tourney.apply(
    lambda r: f"({r['Seed1']}) {r['Team1']}" if r['Pred'] > 0.5
              else f"({r['Seed2']}) {r['Team2']}", axis=1)
tourney['FavoredPct'] = tourney['Pred'].apply(lambda p: f"{max(p, 1-p):.1%}")
tourney['Closeness']  = (tourney['Pred'] - 0.5).abs()

print(f"Total 2026 tournament matchups: {len(tourney):,}")


In [ ]:
# Most confident men's matchups
tourney_m = tourney[tourney['Team1ID'] < 2000].sort_values('Pred', ascending=False)
print("── TOP 20 MEN'S (most lopsided) ──────────────────────────────────────")
for _, r in tourney_m.head(20).iterrows():
    print(f"  ({r['Seed1']}) {r['Team1']:20s} vs ({r['Seed2']}) {r['Team2']:20s}"
          f"  →  {r['Favored']}  {r['FavoredPct']}")

print("\n── TOP 20 MEN'S (closest matchups) ───────────────────────────────────")
for _, r in tourney_m.nsmallest(20, 'Closeness').iterrows():
    print(f"  ({r['Seed1']}) {r['Team1']:20s} vs ({r['Seed2']}) {r['Team2']:20s}"
          f"  →  {r['Favored']}  {r['FavoredPct']}")
